# 14 — NVIDIA AI Experiments

**FinRAG learning series · Notebook 14 of 14**

## Learning objectives
1. Detect GPUs, CUDA and NVIDIA libraries **before** running any GPU operation.
2. Measure CPU performance for the operations FinRAG uses and understand what a GPU would change.
3. Estimate GPU memory for LLM inference (weights + KV cache) and the effect of quantisation.
4. Know the roles of CUDA, NIM, TensorRT-LLM and NeMo, and how FinRAG connects to them.

## Prerequisites
Notebooks 08, 10 and 13. GPU sections run only if a CUDA GPU is detected. **The saved outputs were produced on a machine without
an NVIDIA GPU**; every GPU-only step below reports that it was skipped. No GPU result in this notebook is simulated.

In [1]:
# Setup: make the production backend importable and keep secrets out of outputs
import sys, os
sys.path.insert(0, os.path.abspath("."))      # run notebooks from the notebooks/ folder
import finrag_nb as nb                          # helper: paths, safe settings, DB session
print("Backend package:", nb.BACKEND_DIR)
print("Documents folder:", nb.DOCS_DIR)

Backend package: /home/user/Ready_To_Deploy/backend
Documents folder: /opt/finrag_testdocs


## 1. Hardware and library detection

In [2]:
import importlib.util, os, platform, shutil, subprocess, torch
print("platform:", platform.platform(), "| CPU cores:", os.cpu_count())
print("torch:", torch.__version__, "| built with CUDA:", torch.version.cuda)
CUDA = torch.cuda.is_available()
print("torch.cuda.is_available():", CUDA)
if CUDA:
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}, {p.total_memory / 1e9:.1f} GB, compute capability {p.major}.{p.minor}")
smi = shutil.which("nvidia-smi")
print("nvidia-smi:", subprocess.run([smi], capture_output=True, text=True).stdout[:400] if smi else "not found")
for lib in ("tensorrt_llm", "nemo", "tensorrt", "vllm"):
    print(f"python package {lib!r} installed:", importlib.util.find_spec(lib) is not None)
print("NVIDIA_API_KEY set in environment:", bool(os.environ.get("NVIDIA_API_KEY")), "(value never printed)")

platform: Linux-6.18.44-fc-v819-x86_64-with-glibc2.39 | CPU cores: 4
torch: 2.15.0+cpu | built with CUDA: None
torch.cuda.is_available(): False
nvidia-smi: not found
python package 'tensorrt_llm' installed: False
python package 'nemo' installed: False
python package 'tensorrt' installed: False
python package 'vllm' installed: False
NVIDIA_API_KEY set in environment: False (value never printed)


FinRAG's status endpoint (`GET /api/status`, *AI Model & RAG Status* page) runs the same checks on the server.

## 2. CPU vs GPU: matrix multiplication
Transformers spend most of their time in matrix multiplications. We measure the CPU and, if present, the GPU.

In [3]:
import time
def gflops(device, n=2048, reps=5, dtype=torch.float32):
    a, b = torch.randn(n, n, device=device, dtype=dtype), torch.randn(n, n, device=device, dtype=dtype)
    (a @ b); torch.cuda.synchronize() if device == "cuda" else None
    t = time.time()
    for _ in range(reps):
        a @ b
    if device == "cuda":
        torch.cuda.synchronize()
    return 2 * n ** 3 * reps / (time.time() - t) / 1e9

print(f"CPU float32 matmul: {gflops('cpu'):.1f} GFLOP/s ({torch.get_num_threads()} threads)")
if CUDA:
    print(f"GPU float32 matmul: {gflops('cuda'):.1f} GFLOP/s")
    print(f"GPU float16 matmul: {gflops('cuda', dtype=torch.float16):.1f} GFLOP/s")
else:
    print("GPU benchmark skipped: no CUDA device detected.")

CPU float32 matmul: 499.6 GFLOP/s (4 threads)
GPU benchmark skipped: no CUDA device detected.


## 3. Embedding throughput (the ingestion bottleneck)

In [4]:
from app.rag import embeddings
texts = ["Revenue from Operations | 2024-25: 9,80,136 | 2023-24: 9,14,472 " * 20] * 64
embeddings.embed_documents(texts[:4])            # warm-up / model load
t = time.time(); embeddings.embed_documents(texts); s = time.time() - t
print(f"device={embeddings.device()}: {len(texts) / s:.1f} long chunks/s")
print("On a CUDA GPU FinRAG uses the GPU automatically (embeddings.device()); not measured here." if not CUDA else "")

device=cpu: 11.8 long chunks/s
On a CUDA GPU FinRAG uses the GPU automatically (embeddings.device()); not measured here.


## 4. Memory planning for LLM inference
GPU memory needed ≈ **weights** + **KV cache** (+ activations/overhead). We compute both from the real model configuration of the LLM
FinRAG uses (Qwen3-4B) and compare with the size of the 4-bit GGUF file actually served.

In [5]:
from huggingface_hub import hf_hub_download
import json
cfg = json.load(open(hf_hub_download("Qwen/Qwen3-4B", "config.json")))
L, kvh, hd = cfg["num_hidden_layers"], cfg["num_key_value_heads"], cfg.get("head_dim") or cfg["hidden_size"] // cfg["num_attention_heads"]
params = 4.02e9      # parameter count stated on the Qwen3-4B model card
for name, bytes_per in (("bf16/fp16", 2), ("int8", 1), ("int4", 0.5)):
    print(f"weights {name:<9}: {params * bytes_per / 1e9:5.1f} GB")
gguf = nb.Path("/opt/models/Qwen3-4B-Q4_K_M.gguf")
if gguf.exists():
    print(f"actual Q4_K_M GGUF file: {gguf.stat().st_size / 1e9:.2f} GB (mixed 4/6-bit + metadata)")
for ctx in (4096, 8192, 32768):
    kv = 2 * L * kvh * hd * ctx * 2          # K and V, fp16
    print(f"KV cache fp16 for {ctx:>6} tokens: {kv / 1e9:.2f} GB  (layers={L}, kv_heads={kvh}, head_dim={hd})")

weights bf16/fp16:   8.0 GB
weights int8     :   4.0 GB
weights int4     :   2.0 GB
actual Q4_K_M GGUF file: 2.50 GB (mixed 4/6-bit + metadata)
KV cache fp16 for   4096 tokens: 0.60 GB  (layers=36, kv_heads=8, head_dim=128)
KV cache fp16 for   8192 tokens: 1.21 GB  (layers=36, kv_heads=8, head_dim=128)
KV cache fp16 for  32768 tokens: 4.83 GB  (layers=36, kv_heads=8, head_dim=128)


Grouped-query attention (8 KV heads instead of 32) keeps the KV cache small. In FinRAG's CPU deployment the llama.cpp server also keeps a
*host-RAM prompt cache*; its default (8 GiB) caused out-of-memory kills during development, so FinRAG caps it (`--cache-ram 1024`).

## 5. Measured inference speed of the configured LLM endpoint

In [6]:
from app.llm import client as llm
db = nb.db_session()
ep = llm.active_endpoint(db); db.close()
st = llm.status(ep)
print({k: st.get(k) for k in ("provider", "model", "reachable", "model_available")})
if st.get("reachable"):
    import httpx
    prompt = "Summarise in one sentence: revenue from operations rose from 9,14,472 to 9,80,136 crore. " * 20
    r = httpx.post(f"{ep.base_url}/chat/completions", timeout=600,
                   json={"model": ep.model, "messages": [{"role": "user", "content": prompt + " /no_think"}], "max_tokens": 48})
    tm = r.json().get("timings", {})
    print(f"prompt: {tm.get('prompt_n')} tokens at {tm.get('prompt_per_second', 0):.1f} tok/s | "
          f"generation: {tm.get('predicted_n')} tokens at {tm.get('predicted_per_second', 0):.1f} tok/s")

{'provider': 'llama.cpp (local)', 'model': 'qwen3-4b-q4_k_m', 'reachable': True, 'model_available': True}


prompt: 672 tokens at 82.9 tok/s | generation: 32 tokens at 8.0 tok/s


## 6. NVIDIA NIM
NIM packages models as containers (or hosted endpoints) exposing an **OpenAI-compatible API** — the same interface FinRAG's
`llm/client.py` uses for llama.cpp. Switching is configuration, not code: *Settings → Language model endpoints*, base URL
`https://integrate.api.nvidia.com/v1`, `api_key_env=NVIDIA_API_KEY`.

In [7]:
import httpx
try:
    r = httpx.get("https://integrate.api.nvidia.com/v1/models", timeout=20)
    ids = [m["id"] for m in r.json().get("data", [])]
    print(f"hosted NIM catalogue reachable: HTTP {r.status_code}, {len(ids)} models listed; e.g. {ids[:3]}")
except Exception as exc:
    print("NIM catalogue not reachable from this machine:", type(exc).__name__)
if os.environ.get("NVIDIA_API_KEY"):
    ep_nim = llm.LLMEndpoint("NIM", "NVIDIA NIM", "https://integrate.api.nvidia.com/v1", "meta/llama-3.1-8b-instruct", os.environ["NVIDIA_API_KEY"], False)
    print(llm.chat([{"role": "user", "content": "Reply with the word ready."}], ep_nim, max_tokens=5)["content"])
else:
    print("NIM inference skipped: NVIDIA_API_KEY is not set (no key was available when these outputs were saved).")

hosted NIM catalogue reachable: HTTP 200, 32 models listed; e.g. ['bigcode/starcoder2-15b', 'fastino/gliner2-pii', 'google/gemma-3-12b-it']
NIM inference skipped: NVIDIA_API_KEY is not set (no key was available when these outputs were saved).


## 7. TensorRT-LLM and NeMo
* **TensorRT-LLM** compiles a model into optimised GPU kernels (fused attention, in-flight batching, FP8/INT4). Its server
  (`trtllm-serve`) exposes an OpenAI-compatible API, so FinRAG uses it like NIM. Requires an NVIDIA GPU.
* **NeMo** is NVIDIA's framework for training/customising models at scale (multi-GPU LoRA/SFT). FinRAG's dataset format
  (`train.jsonl` prompt/completion) can be fed to NeMo; the in-repo trainer uses PEFT/TRL.

In [8]:
if CUDA and importlib.util.find_spec("tensorrt_llm"):
    import tensorrt_llm
    print("TensorRT-LLM", tensorrt_llm.__version__, "available")
else:
    print("TensorRT-LLM experiment skipped:", "no CUDA GPU" if not CUDA else "package not installed")
if importlib.util.find_spec("nemo"):
    import nemo
    print("NeMo", nemo.__version__, "available")
else:
    print("NeMo experiment skipped: package not installed (needs a GPU environment; see docs/NVIDIA.md)")

TensorRT-LLM experiment skipped: no CUDA GPU
NeMo experiment skipped: package not installed (needs a GPU environment; see docs/NVIDIA.md)


## 8. Inference optimisation checklist (what each technique buys)
| Technique | Effect | Where in FinRAG |
|---|---|---|
| Quantisation (int4/int8, GGUF, AWQ, FP8) | 2–4× less memory, faster on memory-bound hardware | Q4_K_M GGUF on CPU |
| Smaller context | Fewer prompt tokens → lower latency | 5 passages, 1,100 chars each |
| Prompt caching | Reuse KV for repeated prefixes | llama.cpp cache (capped) |
| Batching / in-flight batching | Higher throughput for many users | NIM / TensorRT-LLM / vLLM |
| GPU offload | 10–100× faster prompt processing | Use a GPU endpoint |

## Interpretation
* On the machine that produced these outputs there is no NVIDIA GPU, so all GPU, TensorRT-LLM and NeMo experiments were skipped and
  NIM inference was skipped for lack of a key. The CPU measurements are real and explain FinRAG's ~1-minute answer latency.
* The memory arithmetic shows why a 4-bit 4B model fits comfortably in 16 GB while bf16 would be tight with a long context.

## Common errors and troubleshooting
| Symptom | Cause | Fix |
|---|---|---|
| `cuda_available: False` with a GPU present | CPU-only PyTorch wheel | Install a CUDA build (`--index-url https://download.pytorch.org/whl/cu128`) |
| Container cannot see the GPU | NVIDIA Container Toolkit missing | Install it; test `docker run --gpus all nvidia/cuda:12.8.1-base-ubuntu24.04 nvidia-smi` |
| NIM 401 | Missing/invalid API key | Set `NVIDIA_API_KEY` in `.env` (never in notebooks) |
| OOM on GPU | Weights + KV cache exceed memory | Quantise, shorten context, smaller model |

## Practical exercises
1. On a GPU machine, re-run sections 2–3 and compare with the saved CPU numbers.
2. Register a NIM endpoint in Settings, *Test* it, and compare answer latency with the CPU model on notebook 10's question.
3. Compute the KV cache for a 32k context with an 8B model and decide which GPU you would need.

## Summary
You detected hardware before any GPU work, measured real CPU performance, planned GPU memory from real model configs, and saw how NIM,
TensorRT-LLM and NeMo fit into FinRAG — with every GPU-only step honestly skipped on this machine.

## Further learning
* CUDA: https://developer.nvidia.com/cuda-toolkit · NIM: https://docs.nvidia.com/nim/
* TensorRT-LLM: https://nvidia.github.io/TensorRT-LLM/ · NeMo: https://docs.nvidia.com/nemo-framework/
* FinRAG: `docs/NVIDIA.md`, `backend/Dockerfile.gpu`, `docker-compose.yml` (profile `nim`)